# OCR KTP Backend — Sistem PMB UM Bandung

**Project:** Implementasi Fitur Autofill Data Calon Mahasiswa Berbasis Tesseract OCR

**Pipeline:** Upload → Grayscale → CLAHE → Gaussian Blur → Otsu Binarization → Deskew → Tesseract OCR → Regex Parsing → JSON Response

---
Jalankan semua cell di bawah secara berurutan.

## 1️⃣ Install System Dependencies (Tesseract OCR)

In [ ]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr tesseract-ocr-ind
!tesseract --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package tesseract-ocr-ind.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-ind_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-ind (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-ind (1:4.00~git30-7274cfa-1.1) ...
tesseract 4.1.1
 leptonica-1.82.0
  libgif 5.1.9 : libjpeg 8d (libjpeg-turbo 2.1.1) : libpng 1.6.37 : libtiff 4.3.0 : zlib 1.2.11 : libwebp 1.2.2 : libopenjp2 2.4.0
 Found AVX512BW
 Found AVX512F
 Found AVX2
 Found AVX
 Found FMA
 Found SSE
 Found libarchive 3.6.0 zlib/1.2.11 liblzma/5.2.5 bz2lib/1.0.8 liblz4/1.9.3 libzstd/1.4.8


## 2️⃣ Install Python Packages

In [ ]:
!pip install -q flask flask-cors pyngrok pytesseract opencv-python-headless Pillow

## 3️⃣ Setup Ngrok Auth Token

Daftar gratis di [https://ngrok.com](https://ngrok.com), lalu salin auth token dari dashboard.

Ganti `YOUR_NGROK_AUTH_TOKEN` di bawah dengan token milikmu.

In [ ]:
NGROK_AUTH_TOKEN = "2ynzLLe0gUOTtBmfekdfSpDH4Mh_845tVELVPcHU3pL5oasTM"
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("✅ Ngrok auth token berhasil di-set!")

✅ Ngrok auth token berhasil di-set!


## 4️⃣ Kode Utama — Flask OCR API

Cell ini berisi seluruh logic backend:
- Preprocessing citra (OpenCV)
- Ekstraksi OCR (Tesseract + Regex)
- REST API endpoint `/api/scan-ktp`

In [ ]:
import re
import time
import threading
import numpy as np
import cv2
import pytesseract
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

# ===================== Flask Init =====================
app = Flask(__name__)
CORS(app, resources={r"/api/*": {"origins": "*", "methods": ["POST", "OPTIONS"], "allow_headers": ["Content-Type"]}})


# =====================================================================
# 1. PREPROCESSING CITRA (OpenCV)
# Pipeline: Decode -> Grayscale -> CLAHE -> GaussianBlur -> Otsu -> Deskew
# =====================================================================

def preprocess_image(image_file):
    """Preprocessing citra KTP untuk optimasi akurasi OCR."""
    file_bytes = np.frombuffer(image_file.read(), np.uint8)
    img = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError("Gagal mendekode gambar. Pastikan file berupa JPG/PNG.")

    # Grayscale: Gray = 0.299R + 0.587G + 0.114B
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # CLAHE (Contrast Limited Adaptive Histogram Equalization)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)

    # Gaussian Blur (noise reduction, kernel 3x3)
    blurred = cv2.GaussianBlur(enhanced, (3, 3), 0)

    # Otsu's Binarization
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Deskew (koreksi kemiringan)
    binary = _deskew(binary)
    return binary


def _deskew(image):
    """Koreksi kemiringan teks menggunakan minAreaRect."""
    coords = np.column_stack(np.where(image > 0))
    if len(coords) < 100:
        return image
    angle = cv2.minAreaRect(coords)[-1]
    if angle < -45:
        angle = -(90 + angle)
    else:
        angle = -angle
    if abs(angle) < 0.5 or abs(angle) > 15:
        return image
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)


# =====================================================================
# 2. EKSTRAKSI DATA KTP (Tesseract OCR + Regex)
# Target: NIK, Nama, Tempat Lahir, Tanggal Lahir, Jenis Kelamin, Alamat
# =====================================================================

def extract_ktp_data(processed_image):
    """Jalankan OCR dan ekstrak 6 field utama dari e-KTP."""
    raw_text = ""
    try:
        raw_text = pytesseract.image_to_string(processed_image, lang="ind+eng", config="--psm 6")
    except Exception:
        try:
            raw_text = pytesseract.image_to_string(processed_image, lang="ind", config="--psm 6")
        except Exception:
            raw_text = pytesseract.image_to_string(processed_image, lang="eng", config="--psm 6")

    confidence = _calc_confidence(processed_image)

    return {
        "nik":           _extract_nik(raw_text),
        "nama":          _extract_nama(raw_text),
        "tempat_lahir":  _extract_tempat_lahir(raw_text),
        "tanggal_lahir": _extract_tanggal_lahir(raw_text),
        "jenis_kelamin": _extract_jenis_kelamin(raw_text),
        "alamat":        _extract_alamat(raw_text),
        "raw_text":      raw_text,
        "confidence":    confidence,
    }


def _calc_confidence(image):
    """Hitung rata-rata confidence score per-word."""
    try:
        data = pytesseract.image_to_data(image, output_type=pytesseract.Output.DICT, config="--psm 6")
        confs = [int(c) for c in data["conf"] if str(c).lstrip("-").isdigit() and int(c) > 0]
        if confs:
            return round(sum(confs) / len(confs), 2)
    except Exception:
        pass
    return 0.0


# ---------- Regex Extraction Functions ----------

def _extract_nik(text):
    """Ekstraksi NIK (16 digit)."""
    try:
        m = re.search(r"NIK\s*[:\-]?\s*(\d[\d\s]{14,19}\d)", text, re.IGNORECASE)
        if m:
            d = re.sub(r"\s", "", m.group(1))
            if len(d) >= 16: return d[:16]
        found = re.findall(r"\b(\d{16})\b", text)
        if found: return found[0]
        relaxed = re.findall(r"(\d[\d\s]{14,19}\d)", text)
        for match in relaxed:
            clean = re.sub(r"\s", "", match)
            if len(clean) >= 16: return clean[:16]
    except Exception: pass
    return "Tidak Terbaca"


def _extract_nama(text):
    """Ekstraksi Nama Lengkap."""
    try:
        m = re.search(r"(?:^|\n)\s*Nama\s*[:\-]?\s*(.+)", text, re.IGNORECASE)
        if m:
            nama = re.sub(r"[^A-Za-z\s'\.\,]", "", m.group(1)).strip()
            if len(nama) >= 2: return nama.upper()
        skip = ["PROVINSI","KABUPATEN","KOTA","KELAMIN","ALAMAT","AGAMA","PEKERJAAN","STATUS","KEWARGANEGARAAN","BERLAKU","NIK"]
        for line in text.split("\n"):
            c = line.strip()
            if c.isupper() and len(c.split())>=2 and not re.search(r"\d",c) and len(c)>4:
                if not any(kw in c for kw in skip): return c
    except Exception: pass
    return "Tidak Terbaca"


def _extract_tempat_lahir(text):
    """Ekstraksi Tempat Lahir dari baris Tempat/Tgl Lahir."""
    try:
        m = re.search(r"[Tt]empat\s*[/\\]?\s*[Tt]gl\.?\s*[Ll]ahir\s*[:\-]?\s*([^,\n]+)", text)
        if m:
            t = re.sub(r"[^A-Za-z\s]", "", m.group(1)).strip()
            if len(t) >= 2: return t.upper()
        fb = re.search(r"([A-Z][A-Za-z\s]{2,20})\s*,\s*\d{2}[-/]\d{2}[-/]\d{4}", text)
        if fb: return fb.group(1).strip().upper()
    except Exception: pass
    return "Tidak Terbaca"


def _extract_tanggal_lahir(text):
    """Ekstraksi Tanggal Lahir (DD-MM-YYYY)."""
    try:
        m = re.search(r"[Tt]empat\s*[/\\]?\s*[Tt]gl\.?\s*[Ll]ahir\s*[:\-]?\s*.+?(\d{2})\s*[-/]\s*(\d{2})\s*[-/]\s*(\d{4})", text)
        if m:
            dd,mm,yyyy = m.group(1),m.group(2),m.group(3)
            if 1<=int(dd)<=31 and 1<=int(mm)<=12 and 1900<=int(yyyy)<=2026:
                return f"{dd}-{mm}-{yyyy}"
        for dd,mm,yyyy in re.findall(r"(\d{2})\s*[-/]\s*(\d{2})\s*[-/]\s*(\d{4})", text):
            if 1<=int(dd)<=31 and 1<=int(mm)<=12 and 1900<=int(yyyy)<=2026:
                return f"{dd}-{mm}-{yyyy}"
    except Exception: pass
    return "Tidak Terbaca"


def _extract_jenis_kelamin(text):
    """Ekstraksi Jenis Kelamin."""
    try:
        up = text.upper()
        if re.search(r"PEREMPUAN|PEREMPU[A-Z]N|WANITA", up): return "PEREMPUAN"
        if re.search(r"LAKI\s*-?\s*LAKI|PRIA", up): return "LAKI-LAKI"
        m = re.search(r"[Jj]enis\s*[Kk]elamin\s*[:\-]?\s*(.+)", text)
        if m:
            v = m.group(1).strip().upper()
            if "PEREMPUAN" in v or "WANITA" in v: return "PEREMPUAN"
            if "LAKI" in v or "PRIA" in v: return "LAKI-LAKI"
    except Exception: pass
    return "Tidak Terbaca"


def _extract_alamat(text):
    """Ekstraksi Alamat (multi-line aware)."""
    try:
        lines = text.split("\n")
        found = False
        parts = []
        for line in lines:
            s = line.strip()
            if re.search(r"Alamat\s*[:\-]?", s, re.IGNORECASE):
                after = re.sub(r".*Alamat\s*[:\-]?\s*", "", s, flags=re.IGNORECASE)
                if after: parts.append(after)
                found = True
                continue
            if found:
                if re.search(r"(RT\s*/\s*RW|Kel[/.]|Kec|Agama|Status|Pekerjaan|Kewarganegaraan)", s, re.IGNORECASE):
                    break
                if s: parts.append(s)
        if parts: return " ".join(parts)
        m = re.search(r"Alamat\s*[:\-]?\s*(.+)", text, re.IGNORECASE)
        if m: return m.group(1).strip()
    except Exception: pass
    return "Tidak Terbaca"


# =====================================================================
# 3. FLASK ROUTES
# =====================================================================

@app.route("/api/scan-ktp", methods=["POST"])
def scan_ktp():
    """POST /api/scan-ktp — Proses citra e-KTP dan kembalikan data JSON."""
    if "ktp_image" not in request.files:
        return jsonify({"status": "error", "message": "Field 'ktp_image' tidak ditemukan."}), 400

    file = request.files["ktp_image"]
    if file.filename == "":
        return jsonify({"status": "error", "message": "Tidak ada file yang dipilih."}), 400

    ext = file.filename.rsplit(".", 1)[-1].lower() if "." in file.filename else ""
    if ext not in {"jpg", "jpeg", "png"}:
        return jsonify({"status": "error", "message": f"Format '{ext}' tidak didukung. Gunakan JPG/PNG."}), 400

    try:
        start = time.time()
        processed = preprocess_image(file)
        result = extract_ktp_data(processed)
        proc_time = round(time.time() - start, 2)

        return jsonify({
            "status":          "success",
            "nik":             result["nik"],
            "nama":            result["nama"],
            "tempat_lahir":    result["tempat_lahir"],
            "tanggal_lahir":   result["tanggal_lahir"],
            "jenis_kelamin":   result["jenis_kelamin"],
            "alamat":          result["alamat"],
            "raw_text":        result["raw_text"],
            "confidence":      result["confidence"],
            "processing_time": proc_time,
        }), 200

    except ValueError as ve:
        return jsonify({"status": "error", "message": str(ve)}), 400
    except Exception as e:
        return jsonify({"status": "error", "message": f"Error: {str(e)}"}), 500


@app.route("/", methods=["GET"])
def health():
    return jsonify({"service": "OCR KTP - PMB UM Bandung", "status": "running"})


print("✅ Semua fungsi berhasil didefinisikan!")

✅ Semua fungsi berhasil didefinisikan!


## 5️⃣ Jalankan Server

Cell ini akan membuka **ngrok tunnel** dan menjalankan Flask server.

⚠️ **PENTING:** Salin URL ngrok yang muncul di output, lalu gunakan URL tersebut di frontend Laravel sebagai pengganti `http://localhost:5000`.

In [ ]:
# Buka ngrok tunnel pada port 5000
public_url = ngrok.connect(5000)

print("=" * 60)
print("\U0001f680 SERVER AKTIF!")
print("=" * 60)
print(f"\n\U0001f310 Public URL (ngrok): {public_url}")
print(f"\U0001f4cc Endpoint OCR:       {public_url}/api/scan-ktp")
print(f"\n\U0001f4cb Salin URL di atas ke frontend Laravel")
print(f"   Ganti 'http://localhost:5000' dengan URL ngrok.")
print("=" * 60)

# Jalankan Flask (blocking)
app.run(port=5000)

🚀 SERVER AKTIF!

🌐 Public URL (ngrok): NgrokTunnel: "https://5e1c-34-124-137-61.ngrok-free.app" -> "http://localhost:5000"
📌 Endpoint OCR:       NgrokTunnel: "https://5e1c-34-124-137-61.ngrok-free.app" -> "http://localhost:5000"/api/scan-ktp

📋 Salin URL di atas ke frontend Laravel
   Ganti 'http://localhost:5000' dengan URL ngrok.
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [03/Aug/2026 12:04:43] "POST /api/scan-ktp HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/Aug/2026 12:11:04] "POST /api/scan-ktp HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/Aug/2026 12:19:19] "POST /api/scan-ktp HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [03/Aug/2026 12:24:58] "POST /api/scan-ktp HTTP/1.1" 200 -
